# ARC-AGI-2 × Qwen3.8-27B-FP8 (4×L4) — Run-All Safe Benchmark

This notebook is designed for **Kaggle Run All / Save & Run All**.

It intentionally loads **one and only one** vLLM TP=4 engine. The old standalone feasibility/model-load cell has been removed, so the previous `Existing llm object detected` / duplicate-engine problem does not occur in a fresh full run.

**Attached Kaggle inputs expected**
- Model: `/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1`
- Offline wheelhouse: `/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3`
- Competition: `/kaggle/input/competitions/arc-prize-2026-arc-agi-2`

Benchmark matrix: batch **1/4/8/16** × generation budget **256/1024/2048/4096**.


In [ ]:
# ==================================================================================================
# CELL 1 — OFFLINE ENVIRONMENT SETUP + INPUT VERIFICATION
# No torch/vLLM model import here. Safe to run first in Run All.
# ==================================================================================================

import os
import sys
import json
import subprocess
import importlib.metadata
from pathlib import Path

MODEL_PATH = Path(
    "/kaggle/input/models/foysalemonshanto/"
    "qwen3-8-27b-fp8-repacked-v1/"
    "pytorch/hf-fp8/1"
)
WHEELHOUSE = Path(
    "/kaggle/input/datasets/driessmit1/"
    "arc3-vllm-h100-wheelhouse-v3"
)
ARC_ROOT = Path(
    "/kaggle/input/competitions/"
    "arc-prize-2026-arc-agi-2"
)

TEST_PATH = ARC_ROOT / "arc-agi_test_challenges.json"
EVAL_PATH = ARC_ROOT / "arc-agi_evaluation_challenges.json"
EVAL_SOL_PATH = ARC_ROOT / "arc-agi_evaluation_solutions.json"
SAMPLE_SUB_PATH = ARC_ROOT / "sample_submission.json"

# Must be set before torch/vLLM is imported.
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("=" * 110)
print("ARC-AGI-2 × Qwen3.8-27B-FP8 | RUN-ALL SAFE SETUP")
print("=" * 110)

for p in [MODEL_PATH, WHEELHOUSE, ARC_ROOT, TEST_PATH, EVAL_PATH, EVAL_SOL_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required Kaggle input is missing: {p}")

# Verify the exact checkpoint metadata that was already proven to load on L4×4.
config_path = MODEL_PATH / "config.json"
index_path = MODEL_PATH / "model.safetensors.index.json"
if not config_path.exists() or not index_path.exists():
    raise FileNotFoundError("Checkpoint config/index is incomplete")

with open(config_path, "r", encoding="utf-8") as f:
    model_cfg = json.load(f)

qcfg = model_cfg.get("quantization_config", {})
print("Model architecture :", model_cfg.get("architectures"))
print("Model type         :", model_cfg.get("model_type"))
print("Quantization       :", qcfg.get("quant_method"), qcfg.get("fmt"), qcfg.get("activation_scheme"))

if model_cfg.get("model_type") != "qwen3_5":
    raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
    raise RuntimeError(f"Unexpected quantization_config={qcfg}")


def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

wanted_vllm = "0.19.0"
current_vllm = pkg_version("vllm")
print("vLLM before setup  :", current_vllm)
print("Torch distribution :", pkg_version("torch"))

# Install only if needed. In the successful feasibility run, torch 2.10.0+cu128 was already present
# and this wheelhouse supplied vLLM 0.19.0 + the compatible dependencies.
if current_vllm != wanted_vllm:
    if "vllm" in sys.modules:
        raise RuntimeError(
            "vLLM is already imported with a different version in this interactive kernel. "
            "Use Kaggle Restart Session, then Run All this notebook."
        )
    print(f"Installing vLLM {wanted_vllm} from offline wheelhouse...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--no-cache-dir",
        "--find-links", str(WHEELHOUSE),
        f"vllm=={wanted_vllm}",
    ])

print("vLLM after setup   :", pkg_version("vllm"))
print("Transformers       :", pkg_version("transformers"))
print("Triton             :", pkg_version("triton"))
print("Setup complete. No model has been loaded yet.")


In [ ]:
# ==================================================================================================
# CELL 2 — ARC SERIALIZATION + TOKEN DISTRIBUTIONS + SAFE CONTEXT SELECTION
# ==================================================================================================

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BATCH_SIZES = [1, 4, 8, 16]
GEN_BUDGETS = [256, 1024, 2048, 4096]
FORCE_FULL_GENERATION = True   # clean throughput benchmark, not final solver behavior
TOTAL_RUNTIME_SEC = 12 * 60 * 60
SAFETY_RESERVE_SEC = 10 * 60
LOCAL_TEST_TASKS = 240

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)
with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)
with open(EVAL_SOL_PATH, "r", encoding="utf-8") as f:
    eval_solutions = json.load(f)
with open(SAMPLE_SUB_PATH, "r", encoding="utf-8") as f:
    sample_submission = json.load(f)

print("Test tasks         :", len(test_data))
print("Evaluation tasks   :", len(eval_data))
print("Evaluation sols    :", len(eval_solutions))
print("Sample submission  :", len(sample_submission))

if len(test_data) != 240 or len(eval_data) != 120 or len(eval_solutions) != 120:
    print("WARNING: task counts differ from the inspected competition snapshot; benchmark will use actual loaded counts.")
LOCAL_TEST_TASKS = len(test_data)

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
)
print("Tokenizer          :", tokenizer.__class__.__name__)
print("Tokenizer vocab    :", len(tokenizer))

SYSTEM_PROMPT = """You are an expert ARC-AGI-2 solver.
Infer the transformation rule from all training input-output examples.
Apply exactly the same transformation to every test input.
Return ONLY valid JSON: {\"test_outputs\":[GRID_0,GRID_1,...]}
Each GRID must be a rectangular 2D integer array using colors 0 through 9.
No explanation. No markdown. No comments. No additional text."""


def grid_to_text(grid):
    return json.dumps(grid, separators=(",", ":"), ensure_ascii=False)


def build_user_prompt(task_id, task):
    parts = [f"TASK_ID={task_id}"]
    for i, ex in enumerate(task["train"]):
        parts += [
            f"TRAIN_{i}_INPUT", grid_to_text(ex["input"]),
            f"TRAIN_{i}_OUTPUT", grid_to_text(ex["output"]),
        ]
    for i, ex in enumerate(task["test"]):
        parts += [f"TEST_{i}_INPUT", grid_to_text(ex["input"])]
    parts.append(f"Return exactly {len(task['test'])} grid(s) in test_outputs.")
    return "\n".join(parts)


def make_prompt(task_id, task):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(task_id, task)},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

# Tokenize every real local test task.
records = []
for task_id, task in test_data.items():
    prompt = make_prompt(task_id, task)
    n_tok = len(tokenizer.encode(prompt, add_special_tokens=False))
    records.append({
        "task_id": task_id,
        "prompt": prompt,
        "prompt_tokens": n_tok,
        "n_train": len(task["train"]),
        "n_test": len(task["test"]),
    })

prompt_lengths = np.array([r["prompt_tokens"] for r in records], dtype=np.int64)
print("\nPROMPT TOKEN DISTRIBUTION")
for p in [0, 10, 25, 50, 75, 90, 95, 99, 100]:
    print(f"P{p:>3}: {np.percentile(prompt_lengths, p):8.1f}")
print(f"Mean: {prompt_lengths.mean():.1f} | Std: {prompt_lengths.std():.1f}")

# Known answer lengths from labeled evaluation tasks.
target_lengths = []
for task_id, solution_grids in eval_solutions.items():
    target_text = json.dumps({"test_outputs": solution_grids}, separators=(",", ":"))
    target_lengths.append(len(tokenizer.encode(target_text, add_special_tokens=False)))
target_lengths = np.array(target_lengths, dtype=np.int64)

print("\nGROUND-TRUTH OUTPUT TOKEN DISTRIBUTION")
for p in [0, 10, 25, 50, 75, 90, 95, 99, 100]:
    print(f"P{p:>3}: {np.percentile(target_lengths, p):8.1f}")
print("Coverage by generation budget:")
for budget in GEN_BUDGETS:
    print(f"  {budget:4d}: {(target_lengths <= budget).mean()*100:6.2f}%")

# Safe context = longest observed prompt + largest tested decode + margin.
max_prompt_tokens = int(prompt_lengths.max())
required_context = max_prompt_tokens + max(GEN_BUDGETS) + 512
MAX_MODEL_LEN = max(18432, int(math.ceil(required_context / 1024) * 1024))
MAX_NUM_BATCHED_TOKENS = 32768

print("\nCONTEXT CONFIGURATION")
print("Longest prompt        :", max_prompt_tokens)
print("Largest generation    :", max(GEN_BUDGETS))
print("Required + margin     :", required_context)
print("Selected max_model_len:", MAX_MODEL_LEN)
print("max_num_batched_tokens:", MAX_NUM_BATCHED_TOKENS)

records_sorted = sorted(records, key=lambda x: x["prompt_tokens"])


def representative_records(n):
    # Evenly sample the entire observed prompt-length range.
    idx = np.linspace(0, len(records_sorted) - 1, n).round().astype(int)
    return [records_sorted[i] for i in idx]

print("\nREPRESENTATIVE BENCHMARK LOADS")
for bs in BATCH_SIZES:
    chosen = representative_records(bs)
    lens = [x["prompt_tokens"] for x in chosen]
    print(
        f"batch={bs:2d} | min={min(lens):5d} | median={np.median(lens):7.1f} "
        f"| max={max(lens):5d} | sum={sum(lens):6d}"
    )

# Save prompt stats before GPU work starts.
prompt_stats_path = Path("/kaggle/working/qwen38_arc_prompt_tokens.csv")
pd.DataFrame([{k: v for k, v in r.items() if k != "prompt"} for r in records]).to_csv(
    prompt_stats_path, index=False
)
print("Saved:", prompt_stats_path)


In [ ]:
# ==================================================================================================
# CELL 3 — LOAD THE TP=4 MODEL ONCE, WARM UP, RUN THE ENTIRE BENCHMARK, SAVE RESULTS
# This is the ONLY LLM() construction in the notebook.
# ==================================================================================================

import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import vllm
from vllm import LLM, SamplingParams

print("=" * 120)
print("Qwen3.8-27B-FP8 × 4 L4 | SINGLE-ENGINE RUN")
print("=" * 120)
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("vLLM    :", vllm.__version__)
print("GPUs    :", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Kaggle 4×L4 accelerator and rerun from a fresh session.")
if torch.cuda.device_count() != 4:
    raise RuntimeError(f"Expected 4 GPUs, but PyTorch sees {torch.cuda.device_count()}.")
for i in range(4):
    cc = torch.cuda.get_device_capability(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {mem:.2f} GiB | CC {cc[0]}.{cc[1]}")

# Crucial fix: no separate feasibility engine exists anywhere earlier in this notebook.
# Therefore Run All creates exactly ONE TP4 engine here.
load_start = time.perf_counter()
arc_llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=4,
    distributed_executor_backend="mp",
    dtype="auto",
    load_format="safetensors",
    trust_remote_code=False,
    language_model_only=True,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=max(BATCH_SIZES),
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    enable_prefix_caching=False,
    seed=SEED,
)
actual_load_sec = time.perf_counter() - load_start
print(f"\nMODEL LOAD COMPLETE: {actual_load_sec:.1f} sec")

# Warmup on a median-size real ARC prompt.
median_record = records_sorted[len(records_sorted) // 2]
warm_params = SamplingParams(temperature=0.0, max_tokens=64, ignore_eos=True)
warm_start = time.perf_counter()
_ = arc_llm.generate([median_record["prompt"]], warm_params, use_tqdm=False)
warm_sec = time.perf_counter() - warm_start
print(f"WARMUP COMPLETE: {warm_sec:.2f} sec")

usable_runtime_sec = TOTAL_RUNTIME_SEC - actual_load_sec - SAFETY_RESERVE_SEC
rows = []
failures = []
experiment_id = 0

print("\n" + "=" * 120)
print("MAIN BENCHMARK: batch 1/4/8/16 × generation 256/1024/2048/4096")
print("=" * 120)

for budget in GEN_BUDGETS:
    for bs in BATCH_SIZES:
        experiment_id += 1
        chosen = representative_records(bs)
        prompts = [r["prompt"] for r in chosen]
        prompt_counts = [r["prompt_tokens"] for r in chosen]
        params = SamplingParams(
            temperature=0.0,
            max_tokens=budget,
            ignore_eos=FORCE_FULL_GENERATION,
        )

        print(
            f"\n[{experiment_id:02d}/{len(GEN_BUDGETS)*len(BATCH_SIZES)}] "
            f"batch={bs:2d} | gen={budget:4d} | prompt_tokens={sum(prompt_counts):6d}"
        )

        try:
            t0 = time.perf_counter()
            outputs = arc_llm.generate(prompts, params, use_tqdm=False)
            wall = time.perf_counter() - t0

            generated_counts = [
                len(o.outputs[0].token_ids) if o.outputs else 0
                for o in outputs
            ]
            n_input = int(sum(prompt_counts))
            n_generated = int(sum(generated_counts))
            tasks_per_sec = bs / wall

            row = {
                "status": "ok",
                "batch": bs,
                "generation": budget,
                "wall_sec": wall,
                "effective_sec_per_task": wall / bs,
                "input_tokens": n_input,
                "generated_tokens": n_generated,
                "avg_generated_tokens_task": float(np.mean(generated_counts)),
                "decode_tok_s": n_generated / wall,
                "total_tok_s": (n_input + n_generated) / wall,
                "tasks_hour": tasks_per_sec * 3600,
                "minutes_240": LOCAL_TEST_TASKS / tasks_per_sec / 60,
                "projected_12h_tasks": usable_runtime_sec * tasks_per_sec,
                "projected_240_passes": usable_runtime_sec * tasks_per_sec / LOCAL_TEST_TASKS,
                "min_prompt": min(prompt_counts),
                "median_prompt": float(np.median(prompt_counts)),
                "max_prompt": max(prompt_counts),
                "error": "",
            }
            rows.append(row)
            print(
                f"  wall={wall:8.2f}s | eff/task={wall/bs:7.2f}s | "
                f"decode={n_generated/wall:8.2f} tok/s | tasks/hr={tasks_per_sec*3600:7.1f} | "
                f"240={LOCAL_TEST_TASKS/tasks_per_sec/60:7.1f} min"
            )

        except Exception as exc:
            # A single aggressive benchmark point should not abort Run All.
            err = f"{type(exc).__name__}: {exc}"
            failures.append({"batch": bs, "generation": budget, "error": err})
            rows.append({
                "status": "failed",
                "batch": bs,
                "generation": budget,
                "wall_sec": np.nan,
                "effective_sec_per_task": np.nan,
                "input_tokens": int(sum(prompt_counts)),
                "generated_tokens": 0,
                "avg_generated_tokens_task": np.nan,
                "decode_tok_s": np.nan,
                "total_tok_s": np.nan,
                "tasks_hour": np.nan,
                "minutes_240": np.nan,
                "projected_12h_tasks": np.nan,
                "projected_240_passes": np.nan,
                "min_prompt": min(prompt_counts),
                "median_prompt": float(np.median(prompt_counts)),
                "max_prompt": max(prompt_counts),
                "error": err,
            })
            print("  BENCHMARK POINT FAILED; continuing Run All:", err)

results_df = pd.DataFrame(rows)
results_path = Path("/kaggle/working/qwen38_arc_throughput_runall.csv")
results_df.to_csv(results_path, index=False)

print("\n" + "=" * 120)
print("RESULTS")
print("=" * 120)
show_cols = [
    "status", "batch", "generation", "wall_sec", "effective_sec_per_task",
    "decode_tok_s", "total_tok_s", "tasks_hour", "minutes_240",
    "projected_12h_tasks", "projected_240_passes"
]
view = results_df[show_cols].copy()
for col in view.columns:
    if col not in ["status", "batch", "generation"]:
        view[col] = pd.to_numeric(view[col], errors="coerce").round(2)
print(view.to_string(index=False))

ok_df = results_df[results_df["status"] == "ok"].copy()
if len(ok_df):
    print("\nBEST SUCCESSFUL CONFIG PER GENERATION BUDGET")
    print("-" * 120)
    best_rows = []
    for budget in GEN_BUDGETS:
        subset = ok_df[ok_df["generation"] == budget]
        if len(subset):
            best_rows.append(subset.loc[subset["tasks_hour"].idxmax()])
    best_df = pd.DataFrame(best_rows)
    best_cols = [
        "generation", "batch", "effective_sec_per_task", "decode_tok_s",
        "tasks_hour", "minutes_240", "projected_12h_tasks", "projected_240_passes"
    ]
    best_view = best_df[best_cols].copy()
    for col in best_view.columns:
        if col not in ["generation", "batch"]:
            best_view[col] = pd.to_numeric(best_view[col], errors="coerce").round(2)
    print(best_view.to_string(index=False))
else:
    best_df = pd.DataFrame()
    print("No benchmark point completed successfully.")

if failures:
    print("\nFAILED POINTS (captured; notebook continued):")
    for x in failures:
        print(f"  batch={x['batch']}, generation={x['generation']}: {x['error']}")

print("\n12-HOUR PROJECTION ASSUMPTIONS")
print("Model load this run :", round(actual_load_sec, 1), "sec")
print("Safety reserve      :", SAFETY_RESERVE_SEC / 60, "min")
print("Usable runtime      :", round(usable_runtime_sec / 3600, 3), "hours")
print("Local task count    :", LOCAL_TEST_TASKS)
print("Saved results       :", results_path)
print("Saved prompt stats  : /kaggle/working/qwen38_arc_prompt_tokens.csv")
print("\nRUN-ALL COMPLETE")
